# Deep Deterministic Policy Gradient (DDPG)

https://spinningup.openai.com/en/latest/algorithms/ddpg.html

In [1]:
import torch
from tensordict.nn import TensorDictModule, TensorDictSequential
from torch import nn
from torchrl import logger
from torchrl.collectors import Collector
from torchrl.data import LazyTensorStorage, ReplayBuffer
from torchrl.envs import Compose, DoubleToFloat, GymEnv, StepCounter, TransformedEnv
from torchrl.modules import (
    AdditiveGaussianModule,
    ProbabilisticActor,
    TanhDelta,
    ValueOperator,
)
from torchrl.objectives import DDPGLoss, SoftUpdate

/usr/local/Caskroom/miniforge/base/envs/trl/lib/python3.13/site-packages/torchrl/modules/mcts/scores.py:574: FutureWarning: functools.partial will be a method descriptor in future Python versions; wrap it in enum.member() if you want to preserve the old behavior
  PUCT = functools.partial(PUCTScore, c=5)  # AlphaGo default value
/usr/local/Caskroom/miniforge/base/envs/trl/lib/python3.13/site-packages/torchrl/modules/mcts/scores.py:575: FutureWarning: functools.partial will be a method descriptor in future Python versions; wrap it in enum.member() if you want to preserve the old behavior
  UCB = functools.partial(UCBScore, c=math.sqrt(2))  # default from Auer et al. 2002
/usr/local/Caskroom/miniforge/base/envs/trl/lib/python3.13/site-packages/torchrl/modules/mcts/scores.py:576: FutureWarning: functools.partial will be a method descriptor in future Python versions; wrap it in enum.member() if you want to preserve the old behavior
  UCB1_TUNED = functools.partial(
/usr/local/Caskroom/mini

Use the `InvertedPendulum` environment, which is a continuous analogue of the `CartPole` environment.

In [2]:
env = TransformedEnv(
    GymEnv("InvertedPendulum-v5"),
    Compose([DoubleToFloat(), StepCounter()]),
)

In [3]:
_ = env.set_seed(0)
_ = torch.manual_seed(0)

Use a deterministic policy which maps the output of the policy network to an action 
(an element of the continuous action space).

`TanhDelta` is a Dirac delta (with P(x=param) = 1) together with the tanh function to 
restrict the range of the action to the interval determined by the environment's action spec.

In [4]:
NUM_CELLS = 256

policy_net = nn.Sequential(
    nn.LazyLinear(NUM_CELLS),
    nn.Tanh(),
    nn.LazyLinear(NUM_CELLS),
    nn.Tanh(),
    nn.LazyLinear(1),
)

policy_module = ProbabilisticActor(
    TensorDictModule(policy_net, in_keys=["observation"], out_keys=["param"]),
    in_keys=["param"],
    spec=env.action_spec,
    safe=True,
    distribution_class=TanhDelta,
    distribution_kwargs={
        "low": env.action_spec.space.low,  # type: ignore
        "high": env.action_spec.space.high,  # type: ignore
    },
)

The Q-value network maps a pair `(s,a)` of observation and action to the corresponding state-action value `Q(s,a)`.

The observation passes through a first network before the action is added and the pair is passes through a second network which outputs the state-action value. This is because the observation is usually more complex than the action and the first network can act as a form of feature extraction for the observation.

In [5]:
class ActionValueNetwork(nn.Module):
    def __init__(self, observation_net: nn.Module, action_net: nn.Module):
        super().__init__()
        self.observation_net = observation_net
        self.action_net = action_net

    def forward(self, observation: torch.Tensor, action: torch.Tensor) -> torch.Tensor:
        xs = self.observation_net(observation)
        xs = nn.Tanh()(xs)
        xs = torch.cat([xs, action], dim=-1)
        xs = self.action_net(xs)

        return xs


value_net = ActionValueNetwork(
    observation_net=nn.Sequential(
        nn.LazyLinear(NUM_CELLS), nn.Tanh(), nn.LazyLinear(NUM_CELLS)
    ),
    action_net=nn.Sequential(nn.LazyLinear(NUM_CELLS), nn.Tanh(), nn.LazyLinear(1)),
)

# ValueOperator = TensorDictModule with out_keys = state_action_value
value_module = ValueOperator(value_net, in_keys=["observation", "action"])

In [6]:
_ = value_module(env.rollout(10, policy_module))  # initialize lazy layers

In [7]:
LEARNING_RATE = 1e-4

loss_module = DDPGLoss(actor_network=policy_module, value_network=value_module)

updater = SoftUpdate(loss_module, eps=0.99)

optimizer = torch.optim.Adam(loss_module.parameters(), lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, 100_000)

Create a stochastic policy by adding Gaussian noise to the output of the
(deterministic) policy to ensure a certain level of initial exploration.

In [8]:
exploration_module = AdditiveGaussianModule(
    spec=env.action_spec,
    annealing_num_steps=100_000,  # decrease noise over time
    safe=True,
)

exploration_policy = TensorDictSequential([policy_module, exploration_module])

The collector takes random actions for a fixed number of steps (on top of using the exploration policy) 
to ensure that the data is varied enough.

Use a large replay buffer for off-policy algorithms since the policy used to obtain experiences is irrelevant;
the Bellman equation should be satisfied for all transitions.

In [9]:
FRAMES_PER_BATCH = 100
INIT_RANDOM_FRAMES = 5000

collector = Collector(
    env,
    policy=exploration_policy,
    frames_per_batch=FRAMES_PER_BATCH,
    init_random_frames=INIT_RANDOM_FRAMES,
    total_frames=-1,
)

buffer = ReplayBuffer(storage=LazyTensorStorage(max_size=100_000))

The training loop is the same as for DQN.

In [10]:
OPTIM_STEPS = 10
BATCH_SIZE = 128

step_count = 0
episode_count = 0


for idx, data in enumerate(collector, start=1):
    buffer.extend(data)

    step_count += data.numel()
    episode_count += data["next", "done"].sum()

    if len(buffer) < collector.init_random_frames:
        continue

    max_steps = data["next", "step_count"].max()

    if idx % 10 == 0:
        logger.info(f"[{idx:>3}] steps: {max_steps:>3}")

    if max_steps > 200:
        break

    for _ in range(OPTIM_STEPS):
        data_batch = buffer.sample(BATCH_SIZE)
        batch_loss = loss_module(data_batch)

        loss = batch_loss["loss_actor"] + batch_loss["loss_value"]
        loss.backward()

        nn.utils.clip_grad_norm_(loss_module.parameters(), 1.0)
        optimizer.step()
        optimizer.zero_grad()

        updater.step()
        scheduler.step()

    exploration_module.step(data.numel())

logger.info(f"solved after {step_count} steps, {episode_count} episodes")

2026-02-16 11:00:28,162 [torchrl][INFO]    Initialized LazyTensorStorage with torch.Size([100000]) shape [END]
2026-02-16 11:00:34,106 [torchrl][INFO]    [ 50] steps:  16 [END]
2026-02-16 11:00:37,054 [torchrl][INFO]    [ 60] steps:   4 [END]
2026-02-16 11:00:39,993 [torchrl][INFO]    [ 70] steps:   4 [END]
2026-02-16 11:00:42,980 [torchrl][INFO]    [ 80] steps:  18 [END]
2026-02-16 11:00:45,865 [torchrl][INFO]    [ 90] steps:   9 [END]
2026-02-16 11:00:48,843 [torchrl][INFO]    [100] steps:  23 [END]
2026-02-16 11:00:51,795 [torchrl][INFO]    [110] steps:  24 [END]
2026-02-16 11:00:54,804 [torchrl][INFO]    [120] steps:  41 [END]
2026-02-16 11:00:57,865 [torchrl][INFO]    [130] steps:  38 [END]
2026-02-16 11:01:00,981 [torchrl][INFO]    [140] steps:  57 [END]
2026-02-16 11:01:04,228 [torchrl][INFO]    [150] steps:  78 [END]
2026-02-16 11:01:07,512 [torchrl][INFO]    [160] steps:  94 [END]
2026-02-16 11:01:10,670 [torchrl][INFO]    [170] steps:  77 [END]
2026-02-16 11:01:13,804 [torchr